In [3]:
import pandas as pd

team_name_mapping = {
    "Arizona Cardinals": "ARI",
    "Atlanta Falcons": "ATL",
    "Baltimore Ravens": "BAL",
    "Buffalo Bills": "BUF",
    "Carolina Panthers": "CAR",
    "Chicago Bears": "CHI",
    "Cincinnati Bengals": "CIN",
    "Cleveland Browns": "CLE",
    "Dallas Cowboys": "DAL",
    "Denver Broncos": "DEN",
    "Detroit Lions": "DET",
    "Green Bay Packers": "GB",
    "Houston Texans": "HOU",
    "Indianapolis Colts": "IND",
    "Jacksonville Jaguars": "JAX",
    "Kansas City Chiefs": "KC",
    "Miami Dolphins": "MIA",
    "Minnesota Vikings": "MIN",
    "New England Patriots": "NE",
    "New Orleans Saints": "NO",
    "New York Giants": "NYG",
    "New York Jets": "NYJ",
    "Oakland Raiders": "LV",  # Fixed
    "Philadelphia Eagles": "PHI",
    "Pittsburgh Steelers": "PIT",
    "San Diego Chargers": "LAC",
    "San Francisco 49ers": "SF",
    "Seattle Seahawks": "SEA",
    "St. Louis Rams": "LAR",  # Fixed
    "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans": "TEN",
    "Washington Football Team": "WAS"  # Fixed
}

# Function to replace team names in the DataFrame
def replace_team_names(df, mapping):
    df['Team'] = df['Team'].map(mapping)
    return df

# File paths
files = {
    "Half PPR": "../PickleFiles/Half PPR Rankings.pkl",
    "Full PPR": "../PickleFiles/Full PPR Rankings.pkl",
    "Non PPR": "../PickleFiles/Non PPR Rankings.pkl"
}

# Define weights
vbd_weight = 1
adp_weight = 0

# Loop through each file
for key, file in files.items():
    # Load the pickle file
    df = pd.read_pickle(file)
    
    # Get the baseline points for each position
    baseline_qb = df[(df['Position'] == 'QB')]['Final PPG'].iloc[min(11, len(df[df['Position'] == 'QB'])-1)]
    baseline_rb = df[(df['Position'] == 'RB')]['Final PPG'].iloc[min(23, len(df[df['Position'] == 'RB'])-1)]
    baseline_wr = df[(df['Position'] == 'WR')]['Final PPG'].iloc[min(29, len(df[df['Position'] == 'WR'])-1)]
    baseline_te = df[(df['Position'] == 'TE')]['Final PPG'].iloc[min(11, len(df[df['Position'] == 'TE'])-1)]
    
    # Calculate VBD for each player based on their position
    def calculate_vbd(row):
        if row['Position'] == 'QB':
            return row['Final PPG'] - baseline_qb
        elif row['Position'] == 'RB':
            return row['Final PPG'] - baseline_rb
        elif row['Position'] == 'WR':
            return row['Final PPG'] - baseline_wr
        elif row['Position'] == 'TE':
            return row['Final PPG'] - baseline_te
        else:
            return 0
    
    df['VBD'] = df.apply(calculate_vbd, axis=1)
    
    # Normalize VBD and ESPN ADP
    df['Normalized VBD'] = (df['VBD'] - df['VBD'].min()) / (df['VBD'].max() - df['VBD'].min())
    df['Normalized ADP'] = (df['ESPN ADP'] - df['ESPN ADP'].min()) / (df['ESPN ADP'].max() - df['ESPN ADP'].min())
    
    # Calculate the weighted score
    df['Weighted Score'] = (vbd_weight * df['Normalized VBD']) + (adp_weight * (1 - df['Normalized ADP']))
    
    # Sort by weighted score in descending order to get the top players based on the weighted score
    df = df.sort_values(by='Weighted Score', ascending=False)
    
    # Adjust the ranking column
    df['Rank'] = range(1, len(df) + 1)

    df = replace_team_names(df, team_name_mapping)

    df['Position Rank'] = df.groupby('Position').cumcount() + 1
    df['Position'] = df['Position'] + df['Position Rank'].astype(str)
    df.drop(columns=['Position Rank'], inplace=True)
    
    # Save the updated DataFrame to a new pickle file
    output_file = file.replace(".pkl", " with Weighted VBD.pkl")
    df.to_pickle(output_file)

df

,Rank,Name,Team,Position,Final PPG,Bye Week,ESPN ADP,VBD,Normalized VBD,Normalized ADP,Weighted Score
10,1,Ray Davis,BUF,RB1,12.471694,12,209.0,7.254195,0.907304,0.838710,0.907304
14,2,Bucky Irving,TB,RB2,11.378791,11,36.0,6.161292,0.828406,0.141129,0.828406
1,3,Caleb Williams,CHI,QB1,17.762341,7,68.0,6.092918,0.823470,0.270161,0.823470
2,4,Brock Purdy,SF,QB2,17.515076,9,94.0,5.845654,0.805619,0.375000,0.805619
3,5,Drake Maye,NE,QB3,16.687332,14,72.0,5.017910,0.745863,0.286290,0.745863
...,...,...,...,...,...,...,...,...,...,...,...
282,285,River Cracraft,WAS,WR114,1.258599,14,NaN,-3.330646,0.143169,NaN,NaN
283,286,Mo Alie-Cox,IND,TE67,1.215097,14,NaN,-4.331298,0.070930,NaN,NaN
285,287,Demarcus Robinson,SF,WR115,1.129168,9,NaN,-3.460077,0.133825,NaN,NaN
286,288,Dawson Knox,BUF,TE68,1.067475,12,NaN,-4.478921,0.060273,NaN,NaN


In [4]:
import nflreadpy
import pandas as pd
import numpy as np

# Try 2025 first, fall back to 2024
for year in [2025, 2024]:
    try:
        seasonal = nflreadpy.load_player_stats([year], summary_level='reg').to_pandas()
        comparison_year = year
        break
    except Exception:
        continue

seasonal = seasonal[seasonal['season_type'] == 'REG']

# Get player names and positions from load_players (gsis_id matches player_id in stats)
players = nflreadpy.load_players().to_pandas()
player_map = players[['gsis_id', 'display_name', 'position']].drop_duplicates('gsis_id')
actual = seasonal.merge(player_map, left_on='player_id', right_on='gsis_id', how='left')

# Calculate actual PPG for each scoring type
actual['Actual PPG (Non PPR)'] = actual['fantasy_points'] / actual['games']
actual['Actual PPG (Full PPR)'] = actual['fantasy_points_ppr'] / actual['games']
actual['Actual PPG (Half PPR)'] = (actual['fantasy_points'] + actual['receptions'] * 0.5) / actual['games']

# Load predicted rankings
scoring_types = {
    'Full PPR': '../PickleFiles/Full PPR Rankings.pkl',
    'Half PPR': '../PickleFiles/Half PPR Rankings.pkl',
    'Non PPR': '../PickleFiles/Non PPR Rankings.pkl',
}

actual_col_map = {
    'Full PPR': 'Actual PPG (Full PPR)',
    'Half PPR': 'Actual PPG (Half PPR)',
    'Non PPR': 'Actual PPG (Non PPR)',
}

print(f"Comparing predictions vs actual {comparison_year} season stats\n")
print("=" * 70)

for scoring_type, pkl_path in scoring_types.items():
    predicted = pd.read_pickle(pkl_path)[['Name', 'Final PPG']]
    actual_col = actual_col_map[scoring_type]
    actual_subset = actual[['display_name', actual_col]].rename(
        columns={'display_name': 'Name', actual_col: 'Actual PPG'}
    )

    # Merge predicted with actual
    comparison = predicted.merge(actual_subset, on='Name', how='inner')
    comparison['Error'] = comparison['Final PPG'] - comparison['Actual PPG']
    comparison['Abs Error'] = comparison['Error'].abs()

    # Metrics
    mae = comparison['Abs Error'].mean()
    rmse = np.sqrt((comparison['Error'] ** 2).mean())
    corr = comparison['Final PPG'].corr(comparison['Actual PPG'])
    matched = len(comparison)

    print(f"\n{'─' * 70}")
    print(f"  {scoring_type}  |  Matched Players: {matched}")
    print(f"{'─' * 70}")
    print(f"  MAE:  {mae:.2f}  |  RMSE: {rmse:.2f}  |  Correlation: {corr:.3f}")

    # Show top 10 biggest misses
    top_misses = comparison.sort_values('Abs Error', ascending=False).head(10)
    print(f"\n  Top 10 biggest prediction misses:")
    print(top_misses[['Name', 'Final PPG', 'Actual PPG', 'Error']].to_string(index=False))

    # Show top 10 most accurate
    top_accurate = comparison.sort_values('Abs Error').head(10)
    print(f"\n  Top 10 most accurate predictions:")
    print(top_accurate[['Name', 'Final PPG', 'Actual PPG', 'Error']].to_string(index=False))

print(f"\n{'=' * 70}")


Comparing predictions vs actual 2025 season stats


──────────────────────────────────────────────────────────────────────
  Full PPR  |  Matched Players: 284
──────────────────────────────────────────────────────────────────────
  MAE:  4.91  |  RMSE: 6.33  |  Correlation: 0.216

  Top 10 biggest prediction misses:
               Name  Final PPG  Actual PPG      Error
Christian McCaffrey   1.212656   24.505882 -23.293227
    Jonathan Taylor   3.851973   21.311765 -17.459791
      Ja'Marr Chase   2.791472   19.600000 -16.808528
 Jaxon Smith-Njigba   4.676812   21.170588 -16.493776
         Puka Nacua   8.611913   23.437500 -14.825587
  Amon-Ra St. Brown   4.310388   19.058824 -14.748435
       Trey McBride   4.074949   18.582353 -14.507404
      Derrick Henry   2.574873   16.441176 -13.866303
          Ray Davis  17.496035    3.770588  13.725446
        Josh Jacobs   2.199461   15.806667 -13.607206

  Top 10 most accurate predictions:
               Name  Final PPG  Actual PPG     Erro